# Package Qwen3-Embedding for offline Kaggle runs

Generated by `scripts/build_embed_benchmark_notebook.py`.

Run with **Internet ON** (CPU is enough). It writes `/kaggle/working/models/<name>/` and `MODELS.json`.
Then save the output as the dataset `boinhbo/qwen3-embedding`.


In [ ]:
import json
import os
from datetime import datetime

from huggingface_hub import HfApi, snapshot_download

REPOS = ['Qwen/Qwen3-Embedding-0.6B', 'Qwen/Qwen3-Embedding-4B']
OUT = '/kaggle/working/models'
LIMIT_GB = 20  # /kaggle/working size limit
ALLOW = ['*.json', '*.safetensors', 'merges.txt', '*.md', '*.txt']
IGNORE = ['onnx/*', 'openvino/*', 'original/*', '*.gguf', '*.bin', '*.onnx', '*.pt', '*.msgpack', '*.h5']
os.makedirs(OUT, exist_ok=True)

api = HfApi()
manifest = {}
for repo in REPOS:
    info = api.model_info(repo, files_metadata=True)
    print(f'{repo}: sha {info.sha}, created {info.created_at}, last modified {info.last_modified}')
    name = repo.split('/')[-1]
    target = os.path.join(OUT, name)
    snapshot_download(repo_id=repo, revision=info.sha, local_dir=target,
                      allow_patterns=ALLOW, ignore_patterns=IGNORE)
    files = []
    for root, dirs, names in os.walk(target):
        dirs[:] = [d for d in dirs if d != '.cache']  # local_dir metadata, not model files
        for file_name in names:
            if file_name.startswith('.'):
                continue
            path = os.path.join(root, file_name)
            files.append(dict(path=os.path.relpath(path, target), bytes=os.path.getsize(path)))
    manifest[repo] = dict(folder=f'models/{name}', sha=info.sha, created_at=str(info.created_at),
                          last_modified=str(info.last_modified), size_bytes=sum(f['bytes'] for f in files),
                          files=sorted(files, key=lambda f: f['path']),
                          paper='arXiv 2506.05176, published 2025-06-05 (from the model card)')

total_gb = sum(m['size_bytes'] for m in manifest.values()) / 1e9
manifest_doc = dict(downloaded_at=datetime.now().isoformat(timespec='seconds'), total_gb=round(total_gb, 2),
                    limit_gb=LIMIT_GB, models=manifest,
                    rule='open models <= 15B released before 2026-08-01 (OVERVIEW.md)')
with open(os.path.join(OUT, 'MODELS.json'), 'w', encoding='utf-8') as file:
    json.dump(manifest_doc, file, ensure_ascii=False, indent=2)
for repo, m in manifest.items():
    print(f"{repo}: sha {m['sha']}, {m['size_bytes'] / 1e9:.2f} GB, {len(m['files'])} files")
print(f'total {total_gb:.2f} GB of /kaggle/working limit {LIMIT_GB} GB')
assert total_gb < LIMIT_GB, 'output exceeds the /kaggle/working limit'


In [ ]:
for name in ['Qwen3-Embedding-0.6B', 'Qwen3-Embedding-4B']:
    folder = os.path.join(OUT, name)
    names = sorted(os.listdir(folder))
    print(name, names)
    with open(os.path.join(folder, 'config.json'), encoding='utf-8') as file:
        config = json.load(file)
    print('  architectures', config.get('architectures'), '| hidden_size', config.get('hidden_size'),
          '| layers', config.get('num_hidden_layers'), '| max_position_embeddings', config.get('max_position_embeddings'))
    assert any(n.endswith('.safetensors') for n in names) and 'tokenizer.json' in names
